> **TrustBreast — Notebook 6 (portability check).** Section 4.10: the same leakage-free pipeline on the Breast Cancer Coimbra cohort. Needs no model files.

# 06 — Pipeline portability: Breast Cancer Coimbra (Section 4.10)

## Chalane ka tareeqa
1. Colab → **Runtime → Change runtime type → CPU** (default; GPU nahi)
2. **Runtime → Run all** (Drive ki zaroorat nahi — data UCI se khud download hota hai)
3. Waqt: takreeban **5–10 minute**
4. **STEP 5 — RESULTS SUMMARY** ka output bhejein.

## Kya hota hai
- 116 patients (64 cancer, 52 controls), 9 anthropometric/blood features.
- WBCD wali pipeline bilkul same: har fold ke andar scaler + SMOTE, RF + XGBoost + DNN, soft voting (0.5), per-fold seeds.
- 5-fold stratified CV (cohort chhota hai), **do dafa** chalti hai — dono runs same aane chahiye.
- Cross-conformal (α = 0.05) out-of-fold probabilities par — marginal aur class-conditional coverage.

In [ ]:
# ---------- STEP 0: determinism ----------
import os
os.environ['PYTHONHASHSEED'] = '42'; os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import random, numpy as np, tensorflow as tf
SEED = 42
tf.keras.utils.set_random_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
print("TF", tf.__version__, "| determinism on")


## STEP 1 — Data (UCI Breast Cancer Coimbra)

In [ ]:
import pandas as pd, numpy as np
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00451/dataR2.csv"
import os
local = next((f for f in ['data/coimbra_dataR2.csv', '../data/coimbra_dataR2.csv', 'dataR2.csv'] if os.path.exists(f)), None)
try:
    df = pd.read_csv(local if local else URL)
    print("Data source:", local if local else URL)
except Exception as e:
    print("Direct download failed, trying ucimlrepo:", str(e)[:60])
    import subprocess; subprocess.run(['pip', '-q', 'install', 'ucimlrepo'])
    from ucimlrepo import fetch_ucirepo
    r = fetch_ucirepo(id=451); df = pd.concat([r.data.features, r.data.targets], axis=1)
target = 'Classification'
X_c = df.drop(columns=[target]).astype(float).values
y_c = (df[target].values == 2).astype(int)          # 1 = breast cancer, 0 = healthy control
FEATS_C = list(df.drop(columns=[target]).columns)
print(f"Coimbra: {len(y_c)} patients | cancer {y_c.sum()} | controls {(y_c == 0).sum()} | features {len(FEATS_C)}: {FEATS_C}")
assert len(y_c) == 116 and y_c.sum() == 64


## STEP 2 — Leakage-free 5-fold CV of the full ensemble (same pipeline as WBCD)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

N_FOLDS = 5
def build_dnn(dim, seed):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(dim,))])
    for u, dr, reg in [(256, .3, 5e-4), (128, .3, 5e-4), (64, .2, None)]:
        m.add(tf.keras.layers.Dense(u, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(reg) if reg else None))
        m.add(tf.keras.layers.BatchNormalization()); m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy'); return m

def run_cv(X, y, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold_acc, oof = [], np.zeros(len(y))
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        sc = MinMaxScaler().fit(X[tr]); Xtr, Xva = sc.transform(X[tr]), sc.transform(X[va])
        Xtr, ytr = SMOTE(random_state=seed).fit_resample(Xtr, y[tr])          # inside the fold only
        rf  = RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=1).fit(Xtr, ytr)
        xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.9,
                            colsample_bytree=0.9, random_state=seed, n_jobs=1,
                            eval_metric='logloss', verbosity=0).fit(Xtr, ytr)
        dnn = build_dnn(Xtr.shape[1], seed + fold)
        dnn.fit(Xtr, ytr, epochs=80, batch_size=16, verbose=0, validation_split=0.15,
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True)])
        p = (rf.predict_proba(Xva)[:, 1] + xgb.predict_proba(Xva)[:, 1] + dnn.predict(Xva, verbose=0).ravel()) / 3
        oof[va] = p; fold_acc.append(accuracy_score(y[va], (p >= .5).astype(int)))
        print(f"  fold {fold}: accuracy {fold_acc[-1]*100:.2f}%")
    return np.array(fold_acc), oof

print("Run 1:"); acc1, oof1 = run_cv(X_c, y_c)
print("Run 2 (reproducibility check):"); acc2, oof2 = run_cv(X_c, y_c)
REPRO = np.array_equal(acc1, acc2) and float(np.max(np.abs(oof1 - oof2))) < 1e-6
print("REPRODUCIBLE ✅" if REPRO else f"NOT reproducible ❌ (max prob diff {np.max(np.abs(oof1-oof2)):.2e})")


## STEP 3 — Metrics (out-of-fold, threshold 0.5)

In [ ]:
pred = (oof1 >= .5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_c, pred).ravel()
rs = np.random.RandomState(42); boot = []
for _ in range(2000):
    b = rs.randint(0, len(y_c), len(y_c)); boot.append((pred[b] == y_c[b]).mean())
M = dict(acc_mean=acc1.mean()*100, acc_sd=acc1.std(ddof=1)*100, pooled=(pred == y_c).mean()*100,
         ci=np.percentile(boot, [2.5, 97.5])*100, auc=roc_auc_score(y_c, oof1), f1=f1_score(y_c, pred),
         sens=tp/(tp+fn), spec=tn/(tn+fp), cm=(tn, fp, fn, tp))
print(f"Accuracy {M['acc_mean']:.2f} ± {M['acc_sd']:.2f} | pooled {M['pooled']:.2f} | CI {M['ci'][0]:.2f}–{M['ci'][1]:.2f}")
print(f"AUC {M['auc']:.4f} | F1 {M['f1']:.4f} | sensitivity {M['sens']:.4f} | specificity {M['spec']:.4f} | TN FP FN TP = {M['cm']}")


## STEP 4 — Cross-conformal coverage (α = 0.05, out-of-fold ensemble probabilities)

In [ ]:
import math
scores = np.where(y_c == 1, 1 - oof1, oof1)
def tau_of(sc, a):
    sc = np.sort(sc); n = len(sc); k = math.ceil((n + 1) * (1 - a)); return sc[min(k, n) - 1], k, n
CC = {}
for a in (0.05, 0.10):
    t, k, n = tau_of(scores, a)
    tb, kb, nb = tau_of(scores[y_c == 0], a); tm, km, nm = tau_of(scores[y_c == 1], a)
    inc0, inc1 = oof1 <= t, (1 - oof1) <= t
    cov = np.mean(np.where(y_c == 1, inc1, inc0)); size = inc0.astype(int) + inc1.astype(int)
    cov_c = np.mean((1 - oof1[y_c == 1]) <= t); cov_h = np.mean(oof1[y_c == 0] <= t)
    mcov_c = np.mean((1 - oof1[y_c == 1]) <= tm); mcov_h = np.mean(oof1[y_c == 0] <= tb)
    CC[a] = dict(t=t, cov=cov, cov_c=cov_c, cov_h=cov_h, tb=tb, tm=tm, mcov_c=mcov_c, mcov_h=mcov_h,
                 sets=(int(np.sum(size == 1)), int(np.sum(size == 2)), int(np.sum(size == 0))),
                 sat=(kb >= nb, km >= nm))
    print(f"alpha={a:.2f}: tau {t:.4f} | marginal {cov*100:.2f}% (cancer {cov_c*100:.2f}%, controls {cov_h*100:.2f}%) "
          f"| sets {CC[a]['sets']} | Mondrian cancer {mcov_c*100:.2f}% controls {mcov_h*100:.2f}% | saturated {CC[a]['sat']}")


## STEP 5 — ★ RESULTS SUMMARY (sirf is cell ka output bhejein)

In [ ]:
c = CC[0.05]
print("="*74 + "\nCOIMBRA PORTABILITY CHECK — Section 4.10\n" + "="*74)
print(f"Reproducible across two runs : {'✅' if REPRO else '❌'}")
print(f"Accuracy (5-fold)            : {M['acc_mean']:.2f} ± {M['acc_sd']:.2f}%   [paper: 70.62 ± 14.35]")
print(f"Pooled out-of-fold accuracy  : {M['pooled']:.2f}%   CI {M['ci'][0]:.2f}–{M['ci'][1]:.2f}   [paper: 70.69, 62.07–79.31]")
print(f"AUC / F1                     : {M['auc']:.2f} / {M['f1']:.2f}   [paper: 0.80 / 0.73]")
print(f"Sensitivity / specificity    : {M['sens']:.2f} / {M['spec']:.2f}   [paper: 0.73 / 0.67]")
print(f"Cross-conformal α=0.05       : marginal {c['cov']*100:.1f}%, cancer {c['cov_c']*100:.1f}%, controls {c['cov_h']*100:.1f}%"
      f"   [paper: 96.6 / 98.4 / 94.2]")
print(f"  Mondrian (class-cond.)     : cancer {c['mcov_c']*100:.1f}%, controls {c['mcov_h']*100:.1f}%  | sets {c['sets']} (single/ambig/empty)")
print("\nNumbers in [paper: ...] are from a Colab CPU-runtime run (Section 4.10); expected prediction sets (41, 75, 0). A GPU runtime can differ in the last decimal.")
